In [1]:
# Для реальной торговли нужно, чтобы:
# 1. Ордеры выставлялись в направлении тренда. Контр-трендовый ордер выставляется только после выполнения трендового.
# 2. После простановки ордера отслеживание текущих цен и моих позиций ведётся через WebSocket, цены подтягиваются автоматически
# 3. Сделка закрывается автоматически по достижении условий
# 4. Stop-loss считается автоматически. В случае, если потери превышают размер SL, сделка автоматически закрывается.

# Для входа в сделку создать отдельное окно, в котором будет изменяющийся график
# Создать кнопку, чтобы, увидев на графике нужные условия, можно было жмякнуть в кнопку и ордеры бы проставились автоматически

# Если цена резко улетает вверх так, что мою позицю по фьючам ликвидирует, просто закрываем спотовую сделку
# Перед входом в сделку проверить, что ни на одной бирже монета не попадает в делистинг
# Перед входом в сделку проверить фандинг. Если он большой - смотрим на время до расчёта и среднее время сделки из истории.
# Если сделка висит и долго не хочет закрываться, можно постепенно начать снижать пороговые значения.

In [2]:
%load_ext autoreload
%autoreload 2

from jaref_bot.data.http_api import ExchangeManager, BybitRestAPI, OKXRestAPI, GateIORestAPI
from jaref_bot.db.db_manager import DBManager
from config import host, user, password, db_name

import pandas as pd
import numpy as np
import logging
from datetime import datetime
from time import sleep
from decimal import Decimal, ROUND_DOWN

from asyncio.exceptions import CancelledError
import aiohttp, asyncio
import nest_asyncio
nest_asyncio.apply()

In [3]:
logging.basicConfig(level=logging.INFO,
                    format="%(message)s")
logging.getLogger('aiohttp').setLevel('ERROR')
logging.getLogger('asyncio').setLevel('ERROR')
logger = logging.getLogger()

In [4]:
def cum_avg_price(order_book, ct_val=1, max_usdt=100):
    """
    Функция рассчитывает средневзвешенную цену по стакану для каждого уровня ордербука.
    :param order_book: Список списков, где каждый вложенный список имеет вид [цена, объём].
                       Например: [[2.5, 10], [2.55, 20], [2.6, 15], [2.65, 5], [2.7, 20]]
    :param ct_val: Размер контракта. Если ct_val != 1, то объём умножается на ct_val.
    :param max_usdt: максимальная сумма сделки. По достижении этого числа цикл прерывается

    :return: Список, состоящий из кортежей вида (avg_price, volume). 
             Пример: [(Decimal('0.02276'), Decimal('5.026')), (Decimal('0.022'), Decimal('7.246'))
    """
    avg_prices = []
    total_volume = 0
    total_cost = 0  # Сумма произведений цена * объём

    for price, volume in order_book:
        # print('before iterations')
        usdt_remain = max_usdt - total_cost
        # print(f'{usdt_remain=}, {total_volume=}, {total_cost=}')
        iter_volume = volume * ct_val
        iter_cost = iter_volume * price
        # print('Добавляем к общим значениям')

        if iter_cost > usdt_remain:
            total_cost += usdt_remain
            total_volume += usdt_remain / price
        else:
            total_cost += iter_cost
            total_volume += iter_volume
        
        avg_price = total_cost / total_volume
        avg_prices.append({'avg_price': avg_price.normalize(), 'usdt_cost': total_cost, 'volume': total_volume})
        # print(f'{usdt_remain=}, {iter_volume=}, {iter_cost=}, {total_volume=}, {total_cost=}')
        if total_cost >= max_usdt:
            break

    return avg_prices

In [5]:
def best_avg_price_from_orderbook(symbol, min_order_usdt, max_usdt, verbose=False):
    best_long_avg_price = np.inf
    best_short_avg_price = 0
    best_long_volume, best_short_volume = 0, 0
    best_short_exc, best_long_exc = None, None
    best_long_dict, best_short_dict = dict(), dict()
    min_order_usdt = 25 # Если в ордер-буке доступно меньше этой суммы, то сделку не рассматриваем
    
    for exc, data in orderbook_dic[symbol].items():
        exchange, market_type = exc.split('_')
        for side, orderbook in data.items():
            try:
                ct_val = coin_information[exc][f'{symbol}_USDT'].get('ct_val', 1)
            except KeyError:
                print(f'Не могу найти инфу для токена {symbol} на бирже {exc}')
                exchange, market_type = exc.split('_')
                if exchange == 'bybit':
                    print('Запрашиваем доп. инфу с биржи ByBit...')
                    coin_information[exc][f'{symbol}_USDT'] = get_missing_instr_info(exchange, market_type, symbol)
                    ct_val = coin_information[exc][f'{symbol}_USDT'].get('ct_val', 1)
                else:
                    raise KeyError
            # if verbose:
            #     print(f'{exc=}, {side=}, {ct_val=}')
            res_dict = cum_avg_price(orderbook, ct_val=ct_val, max_usdt=max_usdt)
            # print(res_dict)
            for level in res_dict:
                avg_price, usdt_cost, volume = level['avg_price'], level['usdt_cost'], level['volume']
                if usdt_cost < min_order_usdt:
                    continue
                # Ищем лучшую среднюю цену для покупки (как на спотовом, так и на фьючерсном рынке)
                if side == 'ask' and avg_price < best_long_avg_price:
                    best_long_avg_price = avg_price
                    best_long_exc = exc
                    best_long_volume = volume
                    best_long_dict = res_dict
    
                # Ищем лучшую цену для шортовой сделки (только фьючерсные рынки)
                if market_type == 'linear' and side == 'bid' and avg_price > best_short_avg_price:
                    best_short_avg_price = avg_price
                    best_short_exc = exc
                    best_short_volume = volume
                    best_short_dict = res_dict
                
            #     if verbose:
            #         print(f'{avg_price=}, {usdt_cost=}, {volume=}')
            # if verbose:
            #     print()

    if verbose:
        print('=========== result ==============')
        print(f'Best long excnahge: {best_long_exc}, avg_price: {best_long_avg_price} and volume: {best_long_volume}')
        # print(f'Long dict:', best_long_dict)
        # print()
        print(f'Best short excnahge: {best_short_exc}, avg_price: {best_short_avg_price} and volume: {best_short_volume}')
        # print(f'Short dict:', best_short_dict)
        print(f'{min_order_usdt=} diff: {(best_short_avg_price / best_long_avg_price - 1) * 100:.2f}')
    
    for x in best_long_dict:
        x['exchange'] = best_long_exc
    for x in best_short_dict:
        x['exchange'] = best_short_exc
    
    return best_long_dict, best_short_dict

In [6]:
def adjust_volume(long_dict, short_dict, min_diff, max_usdt):
    long_total_volume, short_total_volume = 0, 0
    long_total_usdt_cost, short_total_usdt_cost = 0, 0
    long_total_avg_price, short_total_avg_price = 0, 0
    first_iter_flag = True
    
    for long in long_dict:
        for short in short_dict:
            long_avg_price = long['avg_price']
            short_avg_price = short['avg_price']
            long_volume = long['volume']
            short_volume = short['volume']
            long_usdt_cost = long['usdt_cost']
            short_usdt_cost = short['usdt_cost']
    
            if not first_iter_flag:
                curr_diff = (short_total_avg_price / long_total_avg_price - 1) * 100
                # print(f'current diff: {curr_diff:.2f}')
            
            long_total_volume_after_add = long_volume
            short_total_volume_after_add = short_volume
            long_total_usdt_cost_after_add = long_usdt_cost
            short_total_usdt_cost_after_add = short_usdt_cost
            long_total_avg_price_after_add = long_total_usdt_cost_after_add / long_total_volume_after_add
            short_total_avg_price_after_add = short_total_usdt_cost_after_add / short_total_volume_after_add
            diff_after_add = (short_total_avg_price_after_add / long_total_avg_price_after_add - 1) * 100
            
            # print(f'long: avg_price: {long_avg_price:.5f}, usdt_cost: {long_usdt_cost:.5f}, volume: {long_volume}')
            # print(f'short: avg_price: {short_avg_price:.5f}, usdt_cost: {short_usdt_cost:.5f}, volume: {short_volume}')
            # print(f'diff_after_add: {diff_after_add:.2f}')
            # print(f"long_avg: {long_total_avg_price_after_add:.5f}, short_avg: {short_total_avg_price_after_add:.5f}, sum_usdt: {long_total_usdt_cost_after_add}")
    
            if diff_after_add > min_diff and short_total_usdt_cost_after_add <= max_usdt and long_total_usdt_cost_after_add <= max_usdt:
                long_total_volume = long_volume
                short_total_volume = short_volume
                long_total_usdt_cost = long_usdt_cost
                short_total_usdt_cost = short_usdt_cost
                long_total_avg_price = long_total_usdt_cost / long_total_volume
                short_total_avg_price = short_total_usdt_cost / short_total_volume
            else:
                # print(f'Возвращаем: {min(long_total_volume, short_total_volume)}')
                volume = min(long_total_volume, short_total_volume)
                usdt_cost = min(long_total_usdt_cost, short_total_usdt_cost)
                return volume, usdt_cost, curr_diff
    
            first_iter_flag = False
    volume = min(long_total_volume, short_total_volume)
    usdt_cost = min(long_total_usdt_cost, short_total_usdt_cost)
    return volume, usdt_cost, curr_diff

In [7]:
def get_step_info(token, long_exc, short_exc):
    try:
        qty_long_step = coin_information[long_exc][f'{token}_USDT']['qty_step']
    except KeyError:
        arr.append(token)
        print(f'Не могу найти токен "{token}_USDT" для биржи {long_exc}')
    
    try:
        qty_short_step = coin_information[short_exc][f'{token}_USDT']['qty_step']
    except KeyError:
        arr.append(token)
        print(f'Не могу найти токен "{token}_USDT" для биржи {long_exc}')
    
    return max(qty_long_step, qty_short_step)

In [8]:
def round_volume(volume: Decimal, qty_step: Decimal) -> Decimal:
    """
    Округляет значение volume в зависимости от qty_step:
      - Если qty_step == 1: округляем volume до целого (в меньшую сторону).
      - Если qty_step < 1: округляем volume до количества знаков после запятой,
        которое соответствует количеству знаков в qty_step.
      - Если qty_step > 1: округляем volume до ближайшего меньшего числа, кратного qty_step.
    """
    if qty_step == Decimal("1"):
        return Decimal(int(volume))
    elif qty_step < Decimal("1"):
        decimals = -qty_step.as_tuple().exponent
        quantizer = Decimal("1").scaleb(-decimals)
        return volume.quantize(quantizer, rounding=ROUND_DOWN)
    else:
        return (volume // qty_step) * qty_step

In [9]:
async def fetch_all_orderbooks(tokens, limit=10):
    tasks = [exc_manager.get_orderbook(symbol=token, limit=limit) for token in tokens]
    results = await asyncio.gather(*tasks)
    return dict(zip(tokens, results))

In [10]:
def make_list_to_open_position(current_data, edge=0.25):
    # Цену покупки можно брать как с фьючерсов, так и со спотового рынка.
    # Цена продажи должна быть только с фьючерсного рынка.

    # Сейчас в качестве временной меры используем статичный коэф 'edge' для всех монет.
    min_ask_price = current_data.groupby('token')['ask_price'].min()
    linear_df = current_data[current_data['market_type'] == 'linear']
    max_bid_price = linear_df.groupby(['token'])['bid_price'].max()

    df_in = pd.concat([min_ask_price, max_bid_price], axis=1)
    df_in['diff_perc'] = (df_in['bid_price'] / df_in['ask_price'] - 1) * 100
    df_in = df_in[df_in['diff_perc'] > edge].reset_index()
    
    # Здесь будет отбор индивидуальный токенов по результатам бектеста
    # df_in = df_in.merge(spot_params[['token', 'in', 'out', 'mean', 'profit']], on='token')
    # df_in = df_in[df_in['diff_perc'] > df_in['in']]

    return df_in #.sort_values(by='profit', ascending=False).sort_values(by='profit', ascending=False)

In [11]:
def get_missing_instr_info(exchange, market_type, symbol):
    exc_dict = {'bybit': BybitRestAPI, 'okx': OKXRestAPI, 'gate': GateIORestAPI}
    client = exc_dict[exchange](market_type)
    token = client._create_symbol_name(symbol)
    
    return client.get_instrument_data(symbol=token)

In [12]:
def get_volume(orderbook, volume):
    total_usdt = 0
    total_vol = 0

    for price, curr_vol in orderbook:
        if total_vol < volume:
            if curr_vol < volume - total_vol:
                usdt_amount = curr_vol * price
                total_usdt += usdt_amount
                total_vol += curr_vol
                # print(f'Добавляем весь уровень: {curr_vol} по цене {price} = {usdt_amount:.2f}')
                # print(f'{total_usdt=}, {total_vol=}, avg_price: {total_usdt / total_vol:.4f}')
            else:
                # print(f'Объём, который можно добавить: {curr_vol}')
                for _ in range(int(curr_vol)):
                    if total_vol < volume:
                        total_usdt += price
                        total_vol += 1
                        # print(f'Добавляем 1 монету: {total_usdt=}, {total_vol=}, avg_price: {total_usdt / total_vol:.4f}')
                    else:
                        break
                break
            # print(price, curr_vol)
    return total_vol, total_usdt, total_usdt / total_vol

In [13]:
async def place_order(long_exc, short_exc, symbol, volume):
    exc_dict = {'bybit': BybitRestAPI, 'okx': OKXRestAPI, 'gate': GateIORestAPI}
    exchange1, market_type1 = long_exc.split('_')
    exchange2, market_type2 = short_exc.split('_')

    exc_manager = ExchangeManager()
    exc_manager.add_market(long_exc, exc_dict[exchange1](market_type1))
    exc_manager.add_market(short_exc, exc_dict[exchange2](market_type2))

    orders = await exc_manager.get_orderbook(symbol=symbol, limit=20)
    long_vol, long_usdt, long_avg_price = get_volume(orders[long_exc]['ask'], volume=volume)
    short_vol, short_usdt, short_avg_price = get_volume(orders[short_exc]['bid'], volume=volume)

    # long order
    db_manager.add_order(token=symbol, exchange=exchange1, market_type=market_type1, order_type='market', order_id='order_id', 
            order_side='buy', price=long_avg_price, avg_price=long_avg_price, usdt_amount=long_usdt, qty=long_vol, 
            fee=market_fees[long_exc])

    # short order
    db_manager.add_order(token=symbol, exchange=exchange2, market_type=market_type2, order_type='market', order_id='order_id', 
            order_side='sell', price=short_avg_price, avg_price=short_avg_price, usdt_amount=short_usdt, qty=short_vol, 
            fee=market_fees[short_exc])

    print(f"Open order. Buy {long_vol} {symbol} for {long_usdt} with fee {market_fees[long_exc]}")
    print(f"Open order. Sell {short_vol} {symbol} for {short_usdt} with fee {market_fees[short_exc]}")

In [14]:
# top_tokens = ['1INCH', 'ACE', 'ACH', 'ACT', 'AERO', 'AEVO', 'AGLD', 'AI', 'AIOZ', 'AKT', 'ALEO', 'ALGO', 
#               'ALICE', 'ALT', 'ANKR', 'APE', 'API3', 'ARK', 'ARKM', 'ARPA', 'ASTR', 'ATH', 'AUCTION', 'AUDIO', 'AURORA', 
#               'AVAX', 'BAL', 'BANANA', 'BAND', 'BAT', 'BB', 'BICO', 'BIGTIME', 'BLAST', 'BLUR', 'BNX', 'BONK', 'BRETT',
#               'BTT', 'C98', 'CAKE', 'CARV', 'CELO', 'CELR', 'CETUS', 'CHR', 'CKB', 'COMP', 'CORE', 'COTI', 'COW', 
#               'CPOOL', 'CRO', 'CSPR', 'CTC', 'CTK', 'CTSI', 'CVC', 'CVX', 'CYBER', 'DEEP', 'DEGEN', 'DENT', 'DEXE', 'DGB', 
#               'DODO', 'DRIFT', 'DUSK', 'DYM', 'EDU', 'EGLD', 'ENJ', 'ENS', 'ETHFI', 'ETHW', 'FARTCOIN', 'FIDA', 'FLOW', 'FLR', 
#               'FLUX', 'FTN', 'FXS', 'G', 'GAS', 'GIGA', 'GLM', 'GLMR', 'GMT', 'GMX', 'GRASS', 'GRT', 'HFT', 'HIFI', 'HIGH', 
#               'HIPPO', 'HMSTR', 'HNT', 'HOOK', 'HOT', 'ICX', 'ID', 'ILV', 'IMX', 'INJ', 'IOST', 'IOTA', 'IOTX', 'JOE', 'JST', 
#               'JTO', 'JUP', 'KAIA', 'KAVA', 'KDA', 'KNC', 'KSM', 'LADYS', 'LDO', 'LPT', 'LQTY', 'LRC', 'LSK', 'LUMIA', 'LUNC', 
#               'MAGIC', 'MANEKI', 'MANTA', 'MASK', 'MAV', 'MERL', 'METIS', 'MKR', 'MNT', 'MOCA', 'MOG', 'MOODENG', 'MOVR', 'MTL', 
#               'MVL', 'MYRO', 'NEO', 'NMR', 'NOT', 'NTRN', 'OMNI', 'ONE', 'ONG', 'ORBS', 'ORCA', 'OSMO', 
#               'OXT', 'PAAL', 'PENDLE', 'PEOPLE', 'PHA', 'PHB', 'PIXEL', 'PNUT', 'POL', 'POLYX', 'PONKE', 'PORTAL', 'POWR', 'PRIME', 
#               'PYR', 'PYTH', 'QNT', 'QTUM', 'RACA', 'RARE', 'RENDER', 'REQ', 'RIF', 'RLC', 'RON', 'ROSE', 'RPL', 'RSR', 'RSS3', 
#               'RUNE', 'RVN', 'SAFE', 'SAGA', 'SC', 'SCR', 'SCRT', 'SEI', 'SFP', 'SKL', 'SLP', 'SNX', 'SOLO', 'SPEC', 'SPELL', 
#               'SPX', 'SSV', 'STEEM', 'STORJ', 'STPT', 'STX', 'SUN', 'SUPER', 'SUSHI', 'SXP', 'SYN', 'SYS', 'T', 'TAI', 'TAIKO', 'TAO', 
#               'THETA', 'TIA', 'TRB', 'TRU', 'TWT', 'UMA', 'USTC', 'UXLINK', 'VANRY', 'VELO', 'VET', 'VTHO', 'W', 'WAVES', 'WAXP', 
#               'WEMIX', 'WEN', 'WOO', 'X', 'XAI', 'XCH', 'XDC', 'XEC', 'XEM', 'XNO', 'XRD', 'XTZ', 'XVS', 'YFI', 'YGG',  
#               'ZENT', 'ZETA', 'ZIL', 'ZK', 'ZRO', 'ZRX']

# best_tokens = [x + '_USDT' for x in top_tokens]

In [15]:
market_fees = {'bybit_spot': 0.0018, 'bybit_linear': 0.001, 'okx_spot': 0.001, 'okx_linear': 0.0005,
               'gate_spot': 0.002, 'gate_linear': 0.0005}

In [16]:
logger.info('Инициализируем биржи...')
# ====================================
# Инициация нужных криптобирж, рынков и БД
exc_manager = ExchangeManager()
exc_manager.add_market("bybit_linear", BybitRestAPI('linear'))
exc_manager.add_market("bybit_spot", BybitRestAPI('spot'))
exc_manager.add_market("okx_linear", OKXRestAPI('linear'))
exc_manager.add_market("okx_spot", OKXRestAPI('spot'))
exc_manager.add_market("gate_linear", GateIORestAPI('linear'))
exc_manager.add_market("gate_spot", GateIORestAPI('spot'))

db_params = {'host': host, 'user': user, 'password': password, 'database': db_name}
db_manager = DBManager(db_params)

2025-02-08 09:50:31,580 [INFO] Инициализируем биржи...


In [17]:
# Подготовительный этап. Загружаем техническую инфу по токенам (мин. кол-во и число знаков после запятой)
logger.info('Загружаем техническую информацию по токенам...')
coin_information = exc_manager.get_instrument_data()

2025-02-08 09:50:31,628 [INFO] Загружаем техническую информацию по токенам...
2025-02-08 09:50:35,812 [WARNING] Coin LIT_USDT in delisting on Gate.io!


In [18]:
# Подгрузим заранее инфу для тех токенов, которых нет в общей рассылке
logger.info('Подгружаем доп. инфу для тех токенов, которых нет в общей рассылке...')
for token in ('WAXP', 'XEM', 'XNO', 'XCH', 'YFI', 'ZRO', 'ZENT'):
    coin_information['bybit_linear'][f'{token}_USDT'] = get_missing_instr_info('bybit', 'linear', token)

2025-02-08 09:50:37,689 [INFO] Подгружаем доп. инфу для тех токенов, которых нет в общей рассылке...


In [ ]:
logger.info('Запускаем основной цикл программы...')
# ====================================
# Основной цикл
edge = 0.4
usdt_amount = 100
min_usdt_order = 20
max_orders = 10

while True:
    try:
        # Необходимо написать скрипт очистки таблицы, чтобы там не было старых данных.
        current_data = db_manager.get_table('current_data')
        
        # 2. Находим те токены, которые попадают под условия открытия ордера
        df_in = make_list_to_open_position(current_data, edge=edge)
        token_list = df_in['token'].to_list()
        
        # 3. Для этих токенов скачиваем orderbook и подтягиваем техническую инфу (мин. кол-во и дискретность)
        syms = [x.split('_')[0] for x in token_list]
        orderbook_dic = await fetch_all_orderbooks(syms, limit=20)

        for token, data in orderbook_dic.items():
            long_dict, short_dict = best_avg_price_from_orderbook(symbol=token, min_order_usdt=20, max_usdt=usdt_amount, verbose=False)
            diff = (short_dict[0]['avg_price'] / long_dict[0]['avg_price'] - 1) * 100
            
            # if not db_manager.order_exists(token) and n_current_orders < max_orders:
            if diff > edge and not db_manager.order_exists(token):
                long_exc = long_dict[0]['exchange']
                short_exc = short_dict[0]['exchange']
                
                # Обрабатываем идеальный случай, когда можем загрузить полную ставку
                if long_dict[0]['usdt_cost'] == short_dict[0]['usdt_cost'] == usdt_amount:
                    print(f'{datetime.now().strftime('%H:%M:%S')} {token=} diff: {diff:.2f}')
                    long_volume = long_dict[0]['volume']
                    short_volume = short_dict[0]['volume']
                    min_qty_step = get_step_info(token, long_exc, short_exc)
                    
                    volume = min(long_volume, short_volume)
                    volume = round_volume(volume=volume, qty_step=min_qty_step)

                    long_usdt_price = volume * long_dict[0]['avg_price']
                    short_usdt_price = volume * short_dict[0]['avg_price']
                    await place_order(long_exc=long_exc, short_exc=short_exc, symbol=token, volume=volume)
                else:
                    
                    volume, usdt_cost, curr_diff = adjust_volume(long_dict, short_dict, min_diff=edge, max_usdt=usdt_amount)
                    if usdt_cost > min_usdt_order:
                        print(f'{datetime.now().strftime('%H:%M:%S')} {token=} diff: {diff:.2f}')
                        min_qty_step = get_step_info(token, long_exc, short_exc)
                        volume = round_volume(volume=volume, qty_step=min_qty_step)
                        await place_order(long_exc=long_exc, short_exc=short_exc, symbol=token, volume=volume)
        
        sleep(0.2)
    except (KeyboardInterrupt, CancelledError):
        print('Завершение работы.')
        break
    except RuntimeError as e:
        print(f"Ошибка выполнения: {e}")
    
# 4. Если цена всё ещё актуальна, глубина стакана позволяет прибыльно войти в сделку - открываем ордер

2025-02-08 09:51:40,930 [INFO] Запускаем основной цикл программы...


10:04:46 token='STPT' diff: 0.41
Open order. Buy 280 STPT for 21.27560 with fee 0.0005
Open order. Sell 280 STPT for 21.3360 with fee 0.001
